In [10]:
import pandas as pd
import requests
import xml.etree.ElementTree as ET
import time
import re

# 1. Load the initial dataset and extract the names
print("Loading initial dataset...")
df = pd.read_csv('../q/test_scrape.csv')
games_list = df['game'].tolist()

new_dataset = []
print(f"Starting extraction for {len(games_list)} games...\n")

# 2. Define headers to mimic a web browser (Bypasses 401 Unauthorized errors)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
}

# 3. Loop through each game
for original_name in games_list:
    # CLEANING: Remove Wikipedia tags like " (role-playing game)"
    search_name = re.sub(r' \(.*?\)', '', str(original_name)).strip()
    print(f"Processing: {search_name}...")
    
    try:
        # --- PHASE 1: Search API (Get the ID) ---
        # We use boardgamegeek.com instead of rpggeek.com to avoid Cloudflare blocks. 
        # The parameter type=rpgitem ensures we only search for RPGs.
        search_url = "https://boardgamegeek.com/xmlapi2/search"
        search_params = {"query": search_name, "type": "rpgitem", "exact": 1}
        
        search_res = requests.get(search_url, params=search_params, headers=headers)
        time.sleep(2) # MANDATORY sleep to avoid IP bans (429 Too Many Requests)
        
        if search_res.status_code == 200:
            root = ET.fromstring(search_res.content)
            item = root.find('item')
            
            if item is not None:
                game_id = item.get('id')
                
                # --- PHASE 2: Thing API (Get the Stats and Details) ---
                thing_url = "https://boardgamegeek.com/xmlapi2/thing"
                thing_params = {"id": game_id, "stats": 1}
                
                thing_res = requests.get(thing_url, params=thing_params, headers=headers)
                time.sleep(2) # MANDATORY sleep
                
                if thing_res.status_code == 200:
                    t_root = ET.fromstring(thing_res.content)
                    t_item = t_root.find('item')
                    
                    if t_item is not None:
                        # Extract Description
                        desc_tag = t_item.find('description')
                        description = desc_tag.text if desc_tag is not None else "No description"
                        
                        # Extract Average Score
                        average_score = None
                        avg_tag = t_item.find('.//statistics/ratings/average')
                        if avg_tag is not None:
                            average_score = avg_tag.get('value')
                            
                        # Extract Number of Reviews/Ratings
                        num_ratings = None
                        ratings_tag = t_item.find('.//statistics/ratings/usersrated')
                        if ratings_tag is not None:
                            num_ratings = ratings_tag.get('value')
                            
                        # Save the successfully scraped data
                        new_dataset.append({
                            'Name': search_name,
                            'Description': description,
                            'Average Score': average_score,
                            'Number of Reviews': num_ratings
                        })
                        print(f"  -> Success! Score: {average_score} | Reviews: {num_ratings}")
                        continue 
                    else:
                        print("  -> Game details not found in Thing API.")
                else:
                    print(f"  -> Thing API failed. Status Code: {thing_res.status_code}")
            else:
                print("  -> Game not found in Search API results.")
        else:
            print(f"  -> Search API failed. Status Code: {search_res.status_code}")
            
        # If any step above fails normally (e.g. not found), append empty values so we don't lose the row
        new_dataset.append({'Name': search_name, 'Description': None, 'Average Score': None, 'Number of Reviews': None})
            
    except Exception as e:
        print(f"  -> Error processing {search_name}: {e}")
        new_dataset.append({'Name': search_name, 'Description': None, 'Average Score': None, 'Number of Reviews': None})

Loading initial dataset...
Starting extraction for 775 games...

Processing: 13th Age...
  -> Search API failed. Status Code: 401
Processing: The 23rd Letter...
  -> Search API failed. Status Code: 401
Processing: 2300 AD...


KeyboardInterrupt: 

In [ ]:
# 4. Save to a new CSV file
new_df = pd.DataFrame(new_dataset)
new_df.to_csv('ttrpg_api_dataset.csv', index=False)
print("\nExtraction Complete! Data saved to 'ttrpg_api_dataset.csv'")

In [13]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import json
import time

print("Loading initial dataset...")
try:
    df = pd.read_csv('test_scrape.csv')
    games_list = df['game'].tolist()
except FileNotFoundError:
    df = pd.read_csv('../q/test_scrape.csv')
    games_list = df['game'].tolist()

new_dataset = []
print(f"Starting Web Scrape for {len(games_list)} games...\n")

# Headers are critical for scraping to avoid being blocked as a bot
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
}

for original_name in games_list:
    # CLEANING: Remove Wikipedia tags
    search_name = re.sub(r' \(.*?\)', '', str(original_name)).strip()
    print(f"Scraping: {search_name}...")
    
    try:
        # --- PHASE 1: Scrape the Search Page to get the Game's URL ---
        # We use the standard web search URL, not the API
        search_url = "https://boardgamegeek.com/geeksearch.php"
        search_params = {"action": "search", "objecttype": "rpgitem", "q": search_name}
        
        search_res = requests.get(search_url, params=search_params, headers=headers)
        time.sleep(2) # Respect the server
        
        if search_res.status_code == 200:
            soup = BeautifulSoup(search_res.content, 'html.parser')
            
            # Find the first link in the search results that points to an RPG item
            link_tag = soup.find('a', href=re.compile(r'^/rpgitem/'))
            
            if link_tag:
                # Construct the full URL to the game's actual webpage
                item_url = "https://boardgamegeek.com" + link_tag['href']
                
                # --- PHASE 2: Scrape the Game's Webpage ---
                item_res = requests.get(item_url, headers=headers)
                time.sleep(5) 
                
                if item_res.status_code == 200:
                    item_soup = BeautifulSoup(item_res.content, 'html.parser')
                    
                    # 1. Extract Description from the SEO Meta Tags (Easiest HTML scraping)
                    meta_desc = item_soup.find('meta', attrs={'name': 'description'})
                    description = meta_desc['content'] if meta_desc else "No description"
                    
                    # 2. Extract Stats using BeautifulSoup + RegEx!
                    # BGG hides the score in a Javascript variable called GEEK.geekitemPreload
                    average_score = None
                    num_ratings = None
                    
                    # Find all script tags on the page
                    scripts = item_soup.find_all('script')
                    for script in scripts:
                        if script.string and 'GEEK.geekitemPreload' in script.string:
                            # Use RegEx to extract the JSON dictionary hidden inside the Javascript text
                            json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                            
                            if json_match:
                                # Convert the extracted string into a Python dictionary
                                hidden_data = json.loads(json_match.group(1))
                                
                                # Navigate the dictionary to find the stats
                                stats = hidden_data.get('item', {}).get('stats', {})
                                average_score = stats.get('average')
                                num_ratings = stats.get('usersrated')
                            break # We found the data, stop checking other scripts
                            
                    # Save the scraped row
                    new_dataset.append({
                        'Name': search_name,
                        'Description': description,
                        'Average Score': average_score,
                        'Number of Reviews': num_ratings
                    })
                    print(f"  -> Scrape Success! Score: {average_score} | Reviews: {num_ratings}")
                    continue
                else:
                    print(f"  -> Failed to load item page. Status: {item_res.status_code}")
            else:
                print("  -> Game not found in search results.")
        else:
            print(f"  -> Search page failed. Status: {search_res.status_code}")
            
        # Append empty row if failed
        new_dataset.append({'Name': search_name, 'Description': None, 'Average Score': None, 'Number of Reviews': None})
            
    except Exception as e:
        print(f"  -> Error scraping {search_name}: {e}")
        new_dataset.append({'Name': search_name, 'Description': None, 'Average Score': None, 'Number of Reviews': None})



Loading initial dataset...
Starting Web Scrape for 775 games...

Scraping: 13th Age...
  -> Failed to load item page. Status: 403
Scraping: The 23rd Letter...


KeyboardInterrupt: 

In [ ]:
# Save to CSV
new_df = pd.DataFrame(new_dataset)
new_df.to_csv('ttrpg_scraped_dataset.csv', index=False)
print("\nWeb Scrape Complete! Data saved to 'ttrpg_scraped_dataset.csv'")

In [14]:
!pip install cloudscraper

  Using cached cloudscraper-1.2.71-py2.py3-none-any.whl.metadata (19 kB)
Using cached cloudscraper-1.2.71-py2.py3-none-any.whl (99 kB)


In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import re
import json
import time
import cloudscraper # NEW LIBRARY

print("Loading initial dataset...")
try:
    df = pd.read_csv('test_scrape.csv')
    games_list = df['game'].tolist()
except FileNotFoundError:
    df = pd.read_csv('../q/test_scrape.csv')
    games_list = df['game'].tolist()

new_dataset = []
print(f"Starting Web Scrape for {len(games_list)} games...\n")

# 🌟 THE FIX: Create a CloudScraper instance to bypass the 403 Forbidden block
# This perfectly mimics a desktop Chrome browser's fingerprint
scraper = cloudscraper.create_scraper(
    browser={
        'browser': 'chrome',
        'platform': 'windows',
        'desktop': True
    }
)

for original_name in games_list:
    search_name = re.sub(r' \(.*?\)', '', str(original_name)).strip()
    print(f"Scraping: {search_name}...")
    
    try:
        # --- PHASE 1: Scrape Search Page ---
        search_url = "https://boardgamegeek.com/geeksearch.php"
        search_params = {"action": "search", "objecttype": "rpgitem", "q": search_name}
        
        # Use scraper.get instead of requests.get
        search_res = scraper.get(search_url, params=search_params)
        time.sleep(3) # Be polite to the server
        
        if search_res.status_code == 200:
            soup = BeautifulSoup(search_res.content, 'html.parser')
            link_tag = soup.find('a', href=re.compile(r'^/rpgitem/'))
            
            if link_tag:
                item_url = "https://boardgamegeek.com" + link_tag['href']
                
                # --- PHASE 2: Scrape Game Page ---
                item_res = scraper.get(item_url)
                time.sleep(3) 
                
                if item_res.status_code == 200:
                    item_soup = BeautifulSoup(item_res.content, 'html.parser')
                    
                    # 1. Extract Description
                    meta_desc = item_soup.find('meta', attrs={'name': 'description'})
                    description = meta_desc['content'] if meta_desc else "No description"
                    
                    # 2. Extract Stats from JSON inside script
                    average_score = None
                    num_ratings = None
                    
                    scripts = item_soup.find_all('script')
                    for script in scripts:
                        if script.string and 'GEEK.geekitemPreload' in script.string:
                            json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                            if json_match:
                                hidden_data = json.loads(json_match.group(1))
                                stats = hidden_data.get('item', {}).get('stats', {})
                                average_score = stats.get('average')
                                num_ratings = stats.get('usersrated')
                            break 
                            
                    new_dataset.append({
                        'Name': search_name,
                        'Description': description,
                        'Average Score': average_score,
                        'Number of Reviews': num_ratings
                    })
                    print(f"  -> Scrape Success! Score: {average_score} | Reviews: {num_ratings}")
                    continue
                else:
                    print(f"  -> Failed to load item page. Status: {item_res.status_code}")
            else:
                print("  -> Game not found in search results.")
        else:
            print(f"  -> Search page failed. Status: {search_res.status_code}")
            
        new_dataset.append({'Name': search_name, 'Description': None, 'Average Score': None, 'Number of Reviews': None})
            
    except Exception as e:
        print(f"  -> Error scraping {search_name}: {e}")
        new_dataset.append({'Name': search_name, 'Description': None, 'Average Score': None, 'Number of Reviews': None})

new_df = pd.DataFrame(new_dataset)
new_df.to_csv('ttrpg_scraped_dataset.csv', index=False)
print("\nWeb Scrape Complete! Data saved to 'ttrpg_scraped_dataset.csv'")

Loading initial dataset...
Starting Web Scrape for 775 games...

Scraping: 13th Age...
  -> Scrape Success! Score: 7.90114 | Reviews: 131
Scraping: The 23rd Letter...
  -> Scrape Success! Score: 0 | Reviews: 0
Scraping: 2300 AD...
  -> Scrape Success! Score: 7.28917 | Reviews: 60
Scraping: 3D&T...
  -> Scrape Success! Score: 7 | Reviews: 1
Scraping: 7th Sea...
  -> Scrape Success! Score: 7.57759 | Reviews: 116
Scraping: Aberrant...
  -> Scrape Success! Score: 6.96875 | Reviews: 72
Scraping: ACE Agents!...
  -> Scrape Success! Score: 6 | Reviews: 1
Scraping: Active Exploits...
  -> Scrape Success! Score: 5.85 | Reviews: 10
Scraping: Editions of Dungeons & Dragons...
  -> Scrape Success! Score: 6.66667 | Reviews: 3
Scraping: Advanced Fighting Fantasy...
  -> Scrape Success! Score: 7.73913 | Reviews: 23
Scraping: Adventure!...
  -> Scrape Success! Score: 0 | Reviews: 0
Scraping: Adventure!...
  -> Scrape Success! Score: 0 | Reviews: 0
Scraping: Adventurers Guild...
  -> Scrape Success! Sc

In [3]:
new_df.head()

,Name,Description,Average Score,Number of Reviews
0,13th Age,From the back cover:\n\nThe World Won't Save I...,7.90114,131
1,The 23rd Letter,From the publisher:\n\nThe time is now.\n\nPsy...,0,0
2,2300 AD,2300AD is a hard science-fiction role-playing ...,7.28917,60
3,3D&T,from the manual:\nEste guia n&atilde;o inclui ...,7,1
4,7th Sea,The first of two core rulebooks for the 7th Se...,7.57759,116


In [5]:
import pandas as pd
from bs4 import BeautifulSoup
import time
import random
import cloudscraper
import re
import json
import html

print("🚀 Starting the Fault-Tolerant Pure HTML TTRPG Scraper...")

# --- SETUP ---
# Set this to however many pages you want to scrape (100 pages = ~10,000 games)
MAX_PAGES_TO_SCRAPE = 1000
OUTPUT_FILENAME = "ttrpg_database_final.csv"

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

# ==========================================
# PHASE 1: HARVEST GAME LINKS (CORRECTED SESSION)
# ==========================================
import time
import random
from bs4 import BeautifulSoup
import re

game_links = []
MAX_PAGES_TO_SCRAPE = 100 

print(f"\n[PHASE 1] Harvesting {MAX_PAGES_TO_SCRAPE * 100} Game Links...")

# 🌟 THE FIX: We use the 'scraper' object directly. 
# It already acts as a persistent session and keeps your cookies!

for page in range(1, MAX_PAGES_TO_SCRAPE + 1):
    browse_url = f"https://boardgamegeek.com/browse/rpgitem/page/{page}"
    
    # Add a Referer to look like a real browsing path
    if page == 1:
        headers = {"Referer": "https://boardgamegeek.com/"}
    else:
        headers = {"Referer": f"https://boardgamegeek.com/browse/rpgitem/page/{page-1}"}

    print(f"  -> Scraping Browse Page {page:<3}...", end=" ")
    
    try:
        # Use scraper.get directly
        res = scraper.get(browse_url, headers=headers)
        
        # JITTER: Stay between 5 and 9 seconds to be very safe
        time.sleep(random.uniform(5.0, 9.0)) 
        
        if res.status_code == 200:
            # CHECK: Did Cloudflare give us a 'Just a moment' page?
            if "challenge-platform" in res.text or "Just a moment..." in res.text:
                print("| 🚨 Cloudflare Challenge detected! Try manually clearing in browser.")
                break

            soup = BeautifulSoup(res.content, 'html.parser')
            links = soup.find_all('a', href=re.compile(r'^/rpgitem/\d+/'))
            
            if not links:
                print("| 🚨 Zero links found. The page might be blank.")
                break

            for link in links:
                game_links.append(link['href'])
                
            current_unique = len(set(game_links))
            print(f"| 🎯 Total Unique Ready: {current_unique}")
            
            # --- THE COOLDOWN ---
            if page % 5 == 0:
                cooldown = random.randint(20, 40)
                print(f"     ☕ Cooldown... pausing for {cooldown}s...")
                time.sleep(cooldown)
                
        elif res.status_code == 403:
            print(f"| 🚨 403 Forbidden on Page {page}. IP is flagged.")
            break 
        else:
            print(f"| ⚠️ Failed. Status: {res.status_code}")
            
    except Exception as e:
        print(f"| Error: {e}")

game_links = list(set(game_links))
print(f"\n✅ Phase 1 Complete! Passing {len(game_links)} unique games to Phase 2.")

# ==========================================
# PHASE 2: SCRAPE INDIVIDUAL HTML PAGES
# ==========================================
print("\n[PHASE 2] Scraping Game Details (with Auto-Saving)...")
final_dataset = []

for index, link in enumerate(game_links):
    full_url = "https://boardgamegeek.com" + link
    search_name = link.split('/')[-1].replace('-', ' ').title() 
    
    # Print on the same line to keep the terminal clean
    print(f"  -> {index + 1}/{len(game_links)}: {search_name[:20]:<20}...", end=" ")
    
    try:
        res = scraper.get(full_url)
        # Jitter: Random sleep between 2.5 and 4.0 seconds (CRITICAL FOR HTML)
        time.sleep(random.uniform(2.5, 4.0)) 
        
        if res.status_code == 200:
            soup = BeautifulSoup(res.content, 'html.parser')
            
            # --- Extract Description ---
            desc = "No description"
            meta_desc = soup.find('meta', attrs={'name': 'description'})
            if meta_desc:
                desc = meta_desc['content']
                desc = html.unescape(desc)
                desc = re.sub(r"^\s*(From the publisher:|User summary:)\s*", "", desc, flags=re.IGNORECASE)
                desc = re.sub(r'\n+', ' ', desc)
                desc = re.sub(r'\s{2,}', ' ', desc).strip()
                
            # --- Extract Stats & Real Name ---
            avg_score = 0.0
            num_reviews = 0
            
            scripts = soup.find_all('script')
            for script in scripts:
                if script.string and 'GEEK.geekitemPreload' in script.string:
                    json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                    if json_match:
                        try:
                            data = json.loads(json_match.group(1))
                            stats = data.get('item', {}).get('stats', {})
                            avg_score = float(stats.get('average', 0.0))
                            num_reviews = int(stats.get('usersrated', 0))
                            
                            official_name = data.get('item', {}).get('name')
                            if official_name:
                                search_name = official_name
                        except json.JSONDecodeError:
                            pass
                    break 
                    
            final_dataset.append({
                'Name': search_name,
                'Description': desc,
                'Average Score': avg_score,
                'Number of Reviews': num_reviews
            })
            
            print(f"| Score: {avg_score:.2f} | Reviews: {num_reviews}")
            
        elif res.status_code == 429:
            print("| 🚨 Rate Limited! Pausing for 30 seconds...")
            time.sleep(30)
        elif res.status_code == 403:
            print("| 🚨 403 Forbidden! Cloudflare blocked you. Wait 15 mins.")
            time.sleep(30)
        else:
            print(f"| ⚠️ Failed Status: {res.status_code}")
            
    except Exception as e:
        print(f"| Error: {e}")
        
    # --- AUTO-CHECKPOINT SYSTEM ---
    # Every 50 games, save the current progress to the CSV
    if (index + 1) % 50 == 0:
        pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
        print(f"     💾 Checkpoint Reached: Saved {index + 1} rows to {OUTPUT_FILENAME}")

# ==========================================
# FINAL SAVE
# ==========================================
df = pd.DataFrame(final_dataset)
df.to_csv(OUTPUT_FILENAME, index=False)
print(f"\n🎉 FULLY COMPLETE! Scraped {len(df)} games and saved to '{OUTPUT_FILENAME}'")

🚀 Starting the Fault-Tolerant Pure HTML TTRPG Scraper...

[PHASE 1] Harvesting 10000 Game Links...
  -> Scraping Browse Page 1  ... | 🎯 Total Unique Ready: 100
  -> Scraping Browse Page 2  ... | 🎯 Total Unique Ready: 200
  -> Scraping Browse Page 3  ... | 🎯 Total Unique Ready: 300
  -> Scraping Browse Page 4  ... | 🎯 Total Unique Ready: 400
  -> Scraping Browse Page 5  ... | 🎯 Total Unique Ready: 499
     ☕ Cooldown... pausing for 34s...
  -> Scraping Browse Page 6  ... | 🎯 Total Unique Ready: 600
  -> Scraping Browse Page 7  ... | 🎯 Total Unique Ready: 700
  -> Scraping Browse Page 8  ... | 🎯 Total Unique Ready: 800
  -> Scraping Browse Page 9  ... | 🎯 Total Unique Ready: 900
  -> Scraping Browse Page 10 ... | 🎯 Total Unique Ready: 1000
     ☕ Cooldown... pausing for 27s...
  -> Scraping Browse Page 11 ... | 🚨 Zero links found. The page might be blank.

✅ Phase 1 Complete! Passing 1000 unique games to Phase 2.

[PHASE 2] Scraping Game Details (with Auto-Saving)...
  -> 1/1000: The Str

In [6]:
df.head()

,Name,Description,Average Score,Number of Reviews
0,The Strange,Publisher Blurb: Beneath the orbits and atoms ...,7.86486,37
1,InSpectres,Fighting the Forces of Darkness so you don't h...,7.57355,138
2,Dungeons & Dragons Expert Set,The first version of the D&D Expert Set. It co...,7.97881,151
3,The Character Compendium,An unofficial supplement for Warhammer Fantasy...,8.09375,16
4,The Elves of Alfheim,"""This is the first Gazetteer to outline a non-...",7.24878,41


In [ ]:
import os
import time
import random
import re
import json
import html
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper

print("🚀 Starting the Incremental Time-Slice TTRPG Scraper (with Path Pagination)...")

# --- SETUP ---
OUTPUT_FILENAME = "ttrpg_database_final.csv"
START_YEAR = 1980
END_YEAR = 2026

# Initialize the Cloudscraper session
scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

# Human-like headers to spoof a real Chrome browser
human_headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://boardgamegeek.com/advsearch/rpgitem",
    "Sec-Ch-Ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1"
}

# ==========================================
# PHASE 1: HARVEST GAME LINKS (TIME-SLICE BY YEAR)
# ==========================================
game_links = []
print(f"\n[PHASE 1] Harvesting Game Links using Time-Slicing ({START_YEAR} to {END_YEAR})...")

for year in range(START_YEAR, END_YEAR + 1):
    print(f"\n📅 --- Searching Year: {year} ---")
    
    # Loop pages 1 to 10 for each year
    for page in range(1, 11): 
        
        # 🌟 THE FIX: BGG uses path-based pagination (/page/X), NOT query parameters (&page=X) 🌟
        if page == 1:
            search_url = f"https://boardgamegeek.com/search/rpgitem?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
        else:
            search_url = f"https://boardgamegeek.com/search/rpgitem/page/{page}?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
        
        print(f"  -> Scraping Page {page:<2}...", end=" ")
        
        try:
            res = scraper.get(search_url, headers=human_headers)
            
            # Jitter: Random sleep between 5.0 and 8.0 seconds
            time.sleep(random.uniform(5.0, 8.0)) 
            
            if res.status_code == 200:
                soup = BeautifulSoup(res.content, 'html.parser')
                raw_links = soup.find_all('a', href=re.compile(r'^/rpgitem/\d+/'))
                
                if not raw_links:
                    print("| 🚨 End of year reached. Moving to next year.")
                    break 
                
                previous_unique_count = len(set(game_links))
                
                # Deduplicate the raw links on this specific page first
                unique_page_links = set([link['href'] for link in raw_links])
                
                for link in unique_page_links:
                    game_links.append(link)
                    
                current_unique_count = len(set(game_links))
                print(f"| Found {len(unique_page_links):<3} links | 🎯 Total Unique Ready: {current_unique_count}")
                
                # If the count didn't change, BGG is feeding us duplicates! Break the loop!
                if current_unique_count == previous_unique_count:
                    print("     ⏭️ BGG is repeating pages. Moving to next year!")
                    break 
                
            elif res.status_code == 403:
                print("| 🚨 403 Blocked! Cloudflare caught us. (Switch to mobile hotspot!)")
                break 
            else:
                print(f"| ⚠️ Failed. Status: {res.status_code}")
                break
                
        except Exception as e:
            print(f"| Error: {e}")
            
    # --- YEARLY COOLDOWN ---
    cooldown = random.randint(15, 30)
    print(f"     ☕ Year {year} complete. Cooldown for {cooldown}s...")
    time.sleep(cooldown)

# Final clean deduplication before Phase 2
game_links = list(set(game_links))
print(f"\n✅ Phase 1 Complete! Passing {len(game_links)} unique games to Phase 2.")


# ==========================================
# PHASE 2: INCREMENTAL SCRAPING (SMART SKIP)
# ==========================================
print("\n[PHASE 2] Scraping Game Details (with Smart Skip)...")

existing_names = set()
final_dataset = []

# --- LOAD THE EXISTING DATASET ---
if os.path.exists(OUTPUT_FILENAME):
    try:
        existing_df = pd.read_csv(OUTPUT_FILENAME)
        existing_names = set(existing_df['Name'].str.lower().dropna())
        print(f"📦 Successfully loaded {len(existing_names)} existing games to skip.")
        
        final_dataset = existing_df.to_dict('records') 
    except Exception as e:
        print(f"⚠️ Could not load existing dataset: {e}")
else:
    print("⚠️ No existing CSV found. Starting fresh.")

# --- LOOP THROUGH NEW LINKS ---
for index, link in enumerate(game_links):
    full_url = "https://boardgamegeek.com" + link
    
    search_name = link.split('/')[-1].replace('-', ' ').title() 
    
    print(f"  -> {index + 1}/{len(game_links)}: {search_name[:20]:<20}...", end=" ")
    
    # 🌟 THE SMART SKIP 🌟
    if search_name.lower() in existing_names:
        print("| ⏭️ Skipped (Already in CSV)")
        continue
    
    try:
        res = scraper.get(full_url)
        time.sleep(random.uniform(2.5, 4.0)) 
        
        if res.status_code == 200:
            soup = BeautifulSoup(res.content, 'html.parser')
            
            desc = "No description"
            meta_desc = soup.find('meta', attrs={'name': 'description'})
            if meta_desc:
                desc = meta_desc['content']
                desc = html.unescape(desc)
                desc = re.sub(r"^\s*(From the publisher:|User summary:)\s*", "", desc, flags=re.IGNORECASE)
                desc = re.sub(r'\n+', ' ', desc)
                desc = re.sub(r'\s{2,}', ' ', desc).strip()
                
            avg_score = 0.0
            num_reviews = 0
            
            scripts = soup.find_all('script')
            for script in scripts:
                if script.string and 'GEEK.geekitemPreload' in script.string:
                    json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                    if json_match:
                        try:
                            data = json.loads(json_match.group(1))
                            stats = data.get('item', {}).get('stats', {})
                            avg_score = float(stats.get('average', 0.0))
                            num_reviews = int(stats.get('usersrated', 0))
                            
                            official_name = data.get('item', {}).get('name')
                            if official_name:
                                search_name = official_name
                        except json.JSONDecodeError:
                            pass
                    break 
                    
            final_dataset.append({
                'Name': search_name,
                'Description': desc,
                'Average Score': avg_score,
                'Number of Reviews': num_reviews
            })
            
            existing_names.add(search_name.lower())
            print(f"| ✅ Score: {avg_score:.2f} | Reviews: {num_reviews}")
            
        elif res.status_code == 429:
            print("| 🚨 Rate Limited! Pausing for 30 seconds...")
            time.sleep(30)
        elif res.status_code == 403:
            print("| 🚨 403 Forbidden! Cloudflare blocked you.")
            time.sleep(30)
        else:
            print(f"| ⚠️ Failed Status: {res.status_code}")
            
    except Exception as e:
        print(f"| Error: {e}")
        
    # --- AUTO-CHECKPOINT SYSTEM ---
    if (index + 1) % 50 == 0:
        pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
        print(f"     💾 Checkpoint Reached: Saved {len(final_dataset)} total rows to {OUTPUT_FILENAME}")

# ==========================================
# FINAL SAVE
# ==========================================
df = pd.DataFrame(final_dataset)
df.to_csv(OUTPUT_FILENAME, index=False)
print(f"\n🎉 FULLY COMPLETE! Database now has {len(df)} total games in '{OUTPUT_FILENAME}'")

🚀 Starting the Incremental Time-Slice TTRPG Scraper (with Path Pagination)...

[PHASE 1] Harvesting Game Links using Time-Slicing (1980 to 2026)...

📅 --- Searching Year: 1980 ---
  -> Scraping Page 1 ... | Found 100 links | 🎯 Total Unique Ready: 100
  -> Scraping Page 2 ... | Found 54  links | 🎯 Total Unique Ready: 154
  -> Scraping Page 3 ... | 🚨 End of year reached. Moving to next year.
     ☕ Year 1980 complete. Cooldown for 26s...

📅 --- Searching Year: 1981 ---
  -> Scraping Page 1 ... | Found 100 links | 🎯 Total Unique Ready: 254
  -> Scraping Page 2 ... | Found 100 links | 🎯 Total Unique Ready: 354
  -> Scraping Page 3 ... | Found 47  links | 🎯 Total Unique Ready: 401
  -> Scraping Page 4 ... | 🚨 End of year reached. Moving to next year.
     ☕ Year 1981 complete. Cooldown for 29s...

📅 --- Searching Year: 1982 ---
  -> Scraping Page 1 ... | Found 100 links | 🎯 Total Unique Ready: 501
  -> Scraping Page 2 ... | Found 100 links | 🎯 Total Unique Ready: 601
  -> Scraping Page 3 ..

KeyboardInterrupt: 

In [3]:
import os
import time
import random
import re
import json
import html
import math
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper

print("🚀 Starting the Incremental Time-Slice TTRPG Scraper (Dynamic Rolling Quota)...")

# --- SETUP ---
OUTPUT_FILENAME = "ttrpg_database_final.csv"
START_YEAR = 1980
END_YEAR = 2026
TARGET_TOTAL_GAMES = 10000

# Initialize the Cloudscraper session
scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

human_headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://boardgamegeek.com/advsearch/rpgitem",
    "Sec-Ch-Ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1"
}

# ==========================================
# PRE-FLIGHT: LOAD EXISTING DATA
# ==========================================
existing_names = set()
final_dataset = []
games_already_have = 0

if os.path.exists(OUTPUT_FILENAME):
    try:
        existing_df = pd.read_csv(OUTPUT_FILENAME)
        existing_names = set(existing_df['Name'].str.lower().dropna())
        final_dataset = existing_df.to_dict('records') 
        games_already_have = len(final_dataset)
        print(f"📦 Loaded {games_already_have} existing games from CSV.")
    except Exception as e:
        print(f"⚠️ Could not load existing dataset: {e}")
else:
    print("⚠️ No existing CSV found. Starting fresh.")

# Calculate exactly how many we need to scrape in Phase 2
target_needed = TARGET_TOTAL_GAMES - games_already_have
print(f"🎯 Total target remaining: {target_needed} games.")


# ==========================================
# PHASE 1: HARVEST GAME LINKS (DYNAMIC QUOTA)
# ==========================================
game_links = [] 
seen_hrefs = set() 
years_list = list(range(START_YEAR, END_YEAR + 1))

print(f"\n[PHASE 1] Harvesting Links with Rolling Deficit Tracker...")

for i, year in enumerate(years_list):
    if target_needed <= 0:
        print("\n✅ Target links secured! Ending Phase 1 early.")
        break

    remaining_years = len(years_list) - i
    # 🌟 THE FIX: Dynamic Quota calculation 🌟
    # If previous years fell short, this number naturally goes up!
    # Added a +10% buffer to account for skipped duplicates in Phase 2
    current_year_quota = math.ceil((target_needed / remaining_years) * 1.10) 
    
    print(f"\n📅 --- Searching Year: {year} | Target for this year: {current_year_quota} ---")
    year_links_collected = 0
    
    for page in range(1, 11): 
        if year_links_collected >= current_year_quota:
            print(f"  -> 🎯 Reached {current_year_quota} limit for {year}. Moving on.")
            break

        if page == 1:
            search_url = f"https://boardgamegeek.com/search/rpgitem?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
        else:
            search_url = f"https://boardgamegeek.com/search/rpgitem/page/{page}?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
        
        print(f"  -> Scraping Page {page:<2}...", end=" ")
        
        try:
            res = scraper.get(search_url, headers=human_headers)
            time.sleep(random.uniform(4.0, 7.0)) 
            
            if res.status_code == 200:
                soup = BeautifulSoup(res.content, 'html.parser')
                raw_links = soup.find_all('a', href=re.compile(r'^/rpgitem/\d+/'))
                
                if not raw_links:
                    print("| 🚨 End of year reached.")
                    break 
                
                previous_unique_count = len(seen_hrefs)
                unique_page_links = set([link['href'] for link in raw_links])
                
                added_this_page = 0
                for href in unique_page_links:
                    if href not in seen_hrefs and year_links_collected < current_year_quota:
                        seen_hrefs.add(href)
                        game_links.append((href, year)) 
                        year_links_collected += 1
                        added_this_page += 1
                
                print(f"| Added {added_this_page:<3} links | 🎯 Year Total: {year_links_collected}/{current_year_quota}")
                
                if len(seen_hrefs) == previous_unique_count:
                    print("     ⏭️ BGG is repeating pages. Moving to next year!")
                    break 
                
            elif res.status_code == 403:
                print("| 🚨 403 Blocked! Cloudflare caught us.")
                break 
            else:
                print(f"| ⚠️ Failed. Status: {res.status_code}")
                break
                
        except Exception as e:
            print(f"| Error: {e}")
            
    # Deduct the amount we successfully queued up from the overall total
    target_needed -= year_links_collected
    
    cooldown = random.randint(10, 20)
    print(f"     ☕ Year {year} complete. Cooldown for {cooldown}s...")
    print(f"     📊 Total links still needed for 10k: {target_needed if target_needed > 0 else 0}")
    time.sleep(cooldown)

print(f"\n✅ Phase 1 Complete! Passing {len(game_links)} unique game links to Phase 2.")


# ==========================================
# PHASE 2: INCREMENTAL SCRAPING 
# ==========================================
print("\n[PHASE 2] Scraping Game Details...")

for index, (link, year) in enumerate(game_links):
    if len(final_dataset) >= TARGET_TOTAL_GAMES:
        print(f"\n🏁 TARGET REACHED! We have exactly {TARGET_TOTAL_GAMES} games in the dataset.")
        break

    full_url = "https://boardgamegeek.com" + link
    search_name = link.split('/')[-1].replace('-', ' ').title() 
    
    print(f"  -> {index + 1}/{len(game_links)}: [{year}] {search_name[:15]:<15}...", end=" ")
    
    if search_name.lower() in existing_names:
        print("| ⏭️ Skipped (Already in CSV)")
        continue
    
    try:
        res = scraper.get(full_url)
        time.sleep(random.uniform(2.5, 4.0)) 
        
        if res.status_code == 200:
            soup = BeautifulSoup(res.content, 'html.parser')
            
            desc = "No description"
            meta_desc = soup.find('meta', attrs={'name': 'description'})
            if meta_desc:
                desc = meta_desc['content']
                desc = html.unescape(desc)
                desc = re.sub(r"^\s*(From the publisher:|User summary:)\s*", "", desc, flags=re.IGNORECASE)
                desc = re.sub(r'\n+', ' ', desc)
                desc = re.sub(r'\s{2,}', ' ', desc).strip()
                
            avg_score = 0.0
            num_reviews = 0
            
            scripts = soup.find_all('script')
            for script in scripts:
                if script.string and 'GEEK.geekitemPreload' in script.string:
                    json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                    if json_match:
                        try:
                            data = json.loads(json_match.group(1))
                            stats = data.get('item', {}).get('stats', {})
                            avg_score = float(stats.get('average', 0.0))
                            num_reviews = int(stats.get('usersrated', 0))
                            
                            official_name = data.get('item', {}).get('name')
                            if official_name:
                                search_name = official_name
                        except json.JSONDecodeError:
                            pass
                    break 
                    
            final_dataset.append({
                'Name': search_name,
                'Publishing Year': year,
                'Description': desc,
                'Average Score': avg_score,
                'Number of Reviews': num_reviews
            })
            
            existing_names.add(search_name.lower())
            print(f"| ✅ Score: {avg_score:.2f} | Total DB: {len(final_dataset)}")
            
        elif res.status_code == 429:
            print("| 🚨 Rate Limited! Pausing for 30 seconds...")
            time.sleep(30)
        elif res.status_code == 403:
            print("| 🚨 403 Forbidden! Cloudflare blocked you.")
            time.sleep(30)
        else:
            print(f"| ⚠️ Failed Status: {res.status_code}")
            
    except Exception as e:
        print(f"| Error: {e}")
        
    if (index + 1) % 50 == 0:
        pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
        print(f"     💾 Checkpoint Reached: Saved {len(final_dataset)} total rows.")

# ==========================================
# FINAL SAVE
# ==========================================
df = pd.DataFrame(final_dataset)
df.to_csv(OUTPUT_FILENAME, index=False)
print(f"\n🎉 FULLY COMPLETE! Database now has {len(df)} total games in '{OUTPUT_FILENAME}'")

🚀 Starting the Incremental Time-Slice TTRPG Scraper (Dynamic Rolling Quota)...
📦 Loaded 986 existing games from CSV.
🎯 Total target remaining: 9014 games.

[PHASE 1] Harvesting Links with Rolling Deficit Tracker...

📅 --- Searching Year: 1980 | Target for this year: 211 ---
  -> Scraping Page 1 ... | Added 100 links | 🎯 Year Total: 100/211
  -> Scraping Page 2 ... | Added 54  links | 🎯 Year Total: 154/211
  -> Scraping Page 3 ... | 🚨 End of year reached.
     ☕ Year 1980 complete. Cooldown for 17s...
     📊 Total links still needed for 10k: 8860

📅 --- Searching Year: 1981 | Target for this year: 212 ---
  -> Scraping Page 1 ... | Added 100 links | 🎯 Year Total: 100/212
  -> Scraping Page 2 ... | Added 100 links | 🎯 Year Total: 200/212
  -> Scraping Page 3 ... | Added 12  links | 🎯 Year Total: 212/212
  -> 🎯 Reached 212 limit for 1981. Moving on.
     ☕ Year 1981 complete. Cooldown for 12s...
     📊 Total links still needed for 10k: 8648

📅 --- Searching Year: 1982 | Target for this ye

KeyboardInterrupt: 

In [5]:
import os
import time
import random
import re
import json
import html
import math
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper

print("🚀 Starting the Incremental Time-Slice TTRPG Scraper (Clean Slate - Pre-Skip Edition)...")

# --- SETUP ---
OUTPUT_FILENAME = "ttrpg_database_final.csv"
START_YEAR = 1982  # <--- Change this from 1980 to 1982
END_YEAR = 2026
TARGET_TOTAL_GAMES = 10000

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

human_headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://boardgamegeek.com/advsearch/rpgitem",
    "Sec-Ch-Ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1"
}

# ==========================================
# PRE-FLIGHT: LOAD EXISTING DATA
# ==========================================
existing_names = set()
normalized_existing_names = set() # For aggressive Phase 1 pre-skipping
final_dataset = []
games_already_have = 0

if os.path.exists(OUTPUT_FILENAME):
    try:
        existing_df = pd.read_csv(OUTPUT_FILENAME)
        
        # Standard exact match for Phase 2
        existing_names = set(existing_df['Name'].str.lower().dropna())
        
        # Aggressive alphanumeric match for Phase 1 URL checking
        normalized_existing_names = set(
            re.sub(r'[^a-z0-9]', '', str(name).lower()) for name in existing_names
        )
        
        final_dataset = existing_df.to_dict('records') 
        games_already_have = len(final_dataset)
        print(f"📦 Loaded {games_already_have} existing games from CSV.")
    except Exception as e:
        print(f"⚠️ Could not load existing dataset: {e}")
else:
    print("⚠️ No existing CSV found. Starting fresh.")

target_needed = TARGET_TOTAL_GAMES - games_already_have
print(f"🎯 Total target remaining: {target_needed} games.")


# ==========================================
# PHASE 1: HARVEST GAME LINKS (PRE-SKIP DYNAMIC QUOTA)
# ==========================================
game_links = [] 
seen_hrefs = set() 
years_list = list(range(START_YEAR, END_YEAR + 1))

print(f"\n[PHASE 1] Harvesting Links with Pre-Skip Logic...")

for i, year in enumerate(years_list):
    if target_needed <= 0:
        print("\n✅ Target links secured! Ending Phase 1 early.")
        break

    remaining_years = len(years_list) - i
    # 🌟 NEW LOGIC: Calculate quota and add a strict 10% buffer to compensate for edge-cases
    current_year_quota = math.ceil((target_needed / remaining_years) * 1.10) 
    
    print(f"\n📅 --- Searching Year: {year} | Target for this year: {current_year_quota} ---")
    year_links_collected = 0
    
    for page in range(1, 11): 
        if year_links_collected >= current_year_quota:
            print(f"  -> 🎯 Reached {current_year_quota} limit for {year}. Moving on.")
            break

        if page == 1:
            search_url = f"https://boardgamegeek.com/search/rpgitem?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
        else:
            search_url = f"https://boardgamegeek.com/search/rpgitem/page/{page}?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
        
        print(f"  -> Scraping Page {page:<2}...", end=" ")
        
        try:
            res = scraper.get(search_url, headers=human_headers)
            time.sleep(random.uniform(4.0, 7.0)) 
            
            if res.status_code == 200:
                soup = BeautifulSoup(res.content, 'html.parser')
                raw_links = soup.find_all('a', href=re.compile(r'^/rpgitem/\d+/'))
                
                if not raw_links:
                    print("| 🚨 End of year reached.")
                    break 
                
                previous_unique_count = len(seen_hrefs)
                unique_page_links = set([link['href'] for link in raw_links])
                
                added_this_page = 0
                skipped_this_page = 0
                
                for href in unique_page_links:
                    if year_links_collected >= current_year_quota:
                        break

                    if href not in seen_hrefs:
                        # 🌟 PHASE 1 PRE-SKIP: Check URL name against existing DB
                        url_rough_name = re.sub(r'[^a-z0-9]', '', href.split('/')[-1].lower())
                        
                        if url_rough_name in normalized_existing_names:
                            seen_hrefs.add(href)
                            skipped_this_page += 1
                            continue # Do NOT count this toward the quota!
                        
                        seen_hrefs.add(href)
                        game_links.append((href, year)) 
                        year_links_collected += 1
                        added_this_page += 1
                
                print(f"| Added: {added_this_page:<3} | Pre-Skipped: {skipped_this_page:<3} | 🎯 Year Total: {year_links_collected}/{current_year_quota}")
                
                if len(seen_hrefs) == previous_unique_count:
                    print("     ⏭️ BGG is repeating pages. Moving to next year!")
                    break 
                
            elif res.status_code == 403:
                print("| 🚨 403 Blocked! Cloudflare caught us.")
                break 
            else:
                print(f"| ⚠️ Failed. Status: {res.status_code}")
                break
                
        except Exception as e:
            print(f"| Error: {e}")
            
    # Deduct ONLY the highly-likely-to-be-new links from our target
    target_needed -= year_links_collected
    
    cooldown = random.randint(10, 20)
    print(f"     ☕ Year {year} complete. Cooldown for {cooldown}s...")
    print(f"     📊 Total links still needed for 10k: {target_needed if target_needed > 0 else 0}")
    time.sleep(cooldown)

print(f"\n✅ Phase 1 Complete! Passing {len(game_links)} prime game links to Phase 2.")


# ==========================================
# PHASE 2: INCREMENTAL SCRAPING 
# ==========================================
print("\n[PHASE 2] Scraping Game Details...")

for index, (link, year) in enumerate(game_links):
    if len(final_dataset) >= TARGET_TOTAL_GAMES:
        print(f"\n🏁 TARGET REACHED! We have exactly {TARGET_TOTAL_GAMES} games in the dataset.")
        break

    full_url = "https://boardgamegeek.com" + link
    search_name = link.split('/')[-1].replace('-', ' ').title() 
    
    print(f"  -> {index + 1}/{len(game_links)}: [{year}] {search_name[:15]:<15}...", end=" ")
    
    if search_name.lower() in existing_names:
        print("| ⏭️ Skipped (Already in CSV)")
        continue
    
    try:
        res = scraper.get(full_url)
        time.sleep(random.uniform(2.5, 4.0)) 
        
        if res.status_code == 200:
            soup = BeautifulSoup(res.content, 'html.parser')
            
            desc = "No description"
            meta_desc = soup.find('meta', attrs={'name': 'description'})
            if meta_desc:
                desc = meta_desc['content']
                desc = html.unescape(desc)
                desc = re.sub(r"^\s*(From the publisher:|User summary:)\s*", "", desc, flags=re.IGNORECASE)
                desc = re.sub(r'\n+', ' ', desc)
                desc = re.sub(r'\s{2,}', ' ', desc).strip()
                
            avg_score = 0.0
            num_reviews = 0
            
            scripts = soup.find_all('script')
            for script in scripts:
                if script.string and 'GEEK.geekitemPreload' in script.string:
                    json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                    if json_match:
                        try:
                            data = json.loads(json_match.group(1))
                            stats = data.get('item', {}).get('stats', {})
                            avg_score = float(stats.get('average', 0.0))
                            num_reviews = int(stats.get('usersrated', 0))
                            
                            official_name = data.get('item', {}).get('name')
                            if official_name:
                                search_name = official_name
                        except json.JSONDecodeError:
                            pass
                    break 
                    
            final_dataset.append({
                'Name': search_name,
                'Publishing Year': year,
                'Description': desc,
                'Average Score': avg_score,
                'Number of Reviews': num_reviews
            })
            
            existing_names.add(search_name.lower())
            print(f"| ✅ Score: {avg_score:.2f} | Total DB: {len(final_dataset)}")
            
        elif res.status_code == 429:
            print("| 🚨 Rate Limited! Pausing for 30 seconds...")
            time.sleep(30)
        elif res.status_code == 403:
            print("| 🚨 403 Forbidden! Cloudflare blocked you.")
            time.sleep(30)
        else:
            print(f"| ⚠️ Failed Status: {res.status_code}")
            
    except Exception as e:
        print(f"| Error: {e}")
        
    if (index + 1) % 50 == 0:
        pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
        print(f"     💾 Checkpoint Reached: Saved {len(final_dataset)} total rows.")

# ==========================================
# FINAL SAVE
# ==========================================
df = pd.DataFrame(final_dataset)
df.to_csv(OUTPUT_FILENAME, index=False)
print(f"\n🎉 FULLY COMPLETE! Database now has {len(df)} total games in '{OUTPUT_FILENAME}'")

🚀 Starting the Incremental Time-Slice TTRPG Scraper (Clean Slate - Pre-Skip Edition)...
📦 Loaded 1403 existing games from CSV.
🎯 Total target remaining: 8597 games.

[PHASE 1] Harvesting Links with Pre-Skip Logic...

📅 --- Searching Year: 1982 | Target for this year: 211 ---
  -> Scraping Page 1 ... | Added: 17  | Pre-Skipped: 83  | 🎯 Year Total: 17/211
  -> Scraping Page 2 ... | Added: 99  | Pre-Skipped: 1   | 🎯 Year Total: 116/211
  -> Scraping Page 3 ... | Added: 54  | Pre-Skipped: 0   | 🎯 Year Total: 170/211
  -> Scraping Page 4 ... | 🚨 End of year reached.
     ☕ Year 1982 complete. Cooldown for 18s...
     📊 Total links still needed for 10k: 8427

📅 --- Searching Year: 1983 | Target for this year: 211 ---
  -> Scraping Page 1 ... | Added: 82  | Pre-Skipped: 18  | 🎯 Year Total: 82/211
  -> Scraping Page 2 ... | Added: 99  | Pre-Skipped: 1   | 🎯 Year Total: 181/211
  -> Scraping Page 3 ... | Added: 30  | Pre-Skipped: 0   | 🎯 Year Total: 211/211
  -> 🎯 Reached 211 limit for 1983. Mo

KeyboardInterrupt: 

In [1]:
import os
import time
import random
import re
import json
import html
import math
import pandas as pd
from bs4 import BeautifulSoup
import cloudscraper

print("🚀 Starting the Year-by-Year Interleaved TTRPG Scraper (10-Scrape Auto-Save)...")

# --- SETUP ---
OUTPUT_FILENAME = "ttrpg_database_final.csv"
START_YEAR = 1985  # Updated to start at 1985
END_YEAR = 2026
TARGET_TOTAL_GAMES = 10000

scraper = cloudscraper.create_scraper(
    browser={'browser': 'chrome', 'platform': 'windows', 'desktop': True}
)

human_headers = {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://boardgamegeek.com/advsearch/rpgitem",
    "Sec-Ch-Ua": '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
    "Sec-Ch-Ua-Mobile": "?0",
    "Sec-Ch-Ua-Platform": '"Windows"',
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "same-origin",
    "Upgrade-Insecure-Requests": "1"
}

# ==========================================
# PRE-FLIGHT: LOAD EXISTING DATA
# ==========================================
existing_names = set()
normalized_existing_names = set() 
final_dataset = []
seen_hrefs = set() # Global tracker to prevent cross-year duplicate URLs
session_scrapes_count = 0 # Tracks successful Phase 2 scrapes for our 10-save trigger

if os.path.exists(OUTPUT_FILENAME):
    try:
        existing_df = pd.read_csv(OUTPUT_FILENAME)
        existing_names = set(existing_df['Name'].str.lower().dropna())
        
        normalized_existing_names = set(
            re.sub(r'[^a-z0-9]', '', str(name).lower()) for name in existing_names
        )
        
        final_dataset = existing_df.to_dict('records') 
        print(f"📦 Loaded {len(final_dataset)} existing games from CSV.")
    except Exception as e:
        print(f"⚠️ Could not load existing dataset: {e}")
else:
    print("⚠️ No existing CSV found. Starting fresh.")


# ==========================================
# MAIN LOOP: YEAR-BY-YEAR INTERLEAVED SCRAPING
# ==========================================
years_list = list(range(START_YEAR, END_YEAR + 1))

for i, year in enumerate(years_list):
    
    # 1. Calculate precise deficit based on actual saved games
    target_needed = TARGET_TOTAL_GAMES - len(final_dataset)
    
    if target_needed <= 0:
        print(f"\n🏁 TARGET REACHED! We have exactly {TARGET_TOTAL_GAMES} games in the dataset.")
        break

    remaining_years = len(years_list) - i
    current_year_quota = math.ceil((target_needed / remaining_years) * 1.10) 
    
    print(f"\n=======================================================")
    print(f"📅 STARTING YEAR: {year} | Target needed to hit 10k: {target_needed}")
    print(f"=======================================================")
    
    # ---------------------------------------------------------
    # PHASE 1: HARVEST LINKS FOR THIS YEAR ONLY
    # ---------------------------------------------------------
    year_links = [] 
    year_links_collected = 0
    print(f"  [PHASE 1] Harvesting Max {current_year_quota} Links (Pre-Skipping Duplicates)...")
    
    for page in range(1, 11): 
        if year_links_collected >= current_year_quota:
            print(f"    -> 🎯 Reached {current_year_quota} limit for {year}.")
            break

        if page == 1:
            search_url = f"https://boardgamegeek.com/search/rpgitem?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
        else:
            search_url = f"https://boardgamegeek.com/search/rpgitem/page/{page}?advsearch=1&range[yearpublished][min]={year}&range[yearpublished][max]={year}"
        
        print(f"    -> Scraping Page {page:<2}...", end=" ")
        
        try:
            res = scraper.get(search_url, headers=human_headers)
            time.sleep(random.uniform(4.0, 7.0)) 
            
            if res.status_code == 200:
                soup = BeautifulSoup(res.content, 'html.parser')
                raw_links = soup.find_all('a', href=re.compile(r'^/rpgitem/\d+/'))
                
                if not raw_links:
                    print("| 🚨 End of pages for this year.")
                    break 
                
                previous_unique_count = len(seen_hrefs)
                unique_page_links = set([link['href'] for link in raw_links])
                
                added_this_page = 0
                skipped_this_page = 0
                
                for href in unique_page_links:
                    if year_links_collected >= current_year_quota:
                        break

                    if href not in seen_hrefs:
                        url_rough_name = re.sub(r'[^a-z0-9]', '', href.split('/')[-1].lower())
                        
                        if url_rough_name in normalized_existing_names:
                            seen_hrefs.add(href)
                            skipped_this_page += 1
                            continue 
                        
                        seen_hrefs.add(href)
                        year_links.append((href, year)) 
                        year_links_collected += 1
                        added_this_page += 1
                
                print(f"| Added: {added_this_page:<3} | Pre-Skipped: {skipped_this_page:<3} | 🎯 Year Total: {year_links_collected}/{current_year_quota}")
                
                if len(seen_hrefs) == previous_unique_count:
                    print("       ⏭️ BGG is repeating pages. Breaking pagination loop.")
                    break 
                
            elif res.status_code == 403:
                print("| 🚨 403 Blocked! Cloudflare caught us.")
                break 
            else:
                print(f"| ⚠️ Failed. Status: {res.status_code}")
                break
                
        except Exception as e:
            print(f"| Error: {e}")

    # ---------------------------------------------------------
    # PHASE 2: SCRAPE DETAILS FOR THIS YEAR ONLY
    # ---------------------------------------------------------
    if not year_links:
        print(f"  [PHASE 2] No new links found for {year}. Skipping to next year.")
        continue

    print(f"\n  [PHASE 2] Scraping Details for {len(year_links)} new games from {year}...")
    
    for index, (link, game_year) in enumerate(year_links):
        if len(final_dataset) >= TARGET_TOTAL_GAMES:
            print(f"\n🏁 TARGET REACHED DURING PHASE 2! Halting.")
            break

        full_url = "https://boardgamegeek.com" + link
        search_name = link.split('/')[-1].replace('-', ' ').title() 
        
        print(f"    -> {index + 1}/{len(year_links)}: [{game_year}] {search_name[:15]:<15}...", end=" ")
        
        if search_name.lower() in existing_names:
            print("| ⏭️ Skipped (Already in CSV - Caught in Phase 2)")
            continue
        
        try:
            res = scraper.get(full_url)
            time.sleep(random.uniform(2.5, 4.0)) 
            
            if res.status_code == 200:
                soup = BeautifulSoup(res.content, 'html.parser')
                
                desc = "No description"
                meta_desc = soup.find('meta', attrs={'name': 'description'})
                if meta_desc:
                    desc = meta_desc['content']
                    desc = html.unescape(desc)
                    desc = re.sub(r"^\s*(From the publisher:|User summary:)\s*", "", desc, flags=re.IGNORECASE)
                    desc = re.sub(r'\n+', ' ', desc)
                    desc = re.sub(r'\s{2,}', ' ', desc).strip()
                    
                avg_score = 0.0
                num_reviews = 0
                
                scripts = soup.find_all('script')
                for script in scripts:
                    if script.string and 'GEEK.geekitemPreload' in script.string:
                        json_match = re.search(r'GEEK\.geekitemPreload\s*=\s*(\{.*?\});', script.string, re.DOTALL)
                        if json_match:
                            try:
                                data = json.loads(json_match.group(1))
                                stats = data.get('item', {}).get('stats', {})
                                avg_score = float(stats.get('average', 0.0))
                                num_reviews = int(stats.get('usersrated', 0))
                                
                                official_name = data.get('item', {}).get('name')
                                if official_name:
                                    search_name = official_name
                            except json.JSONDecodeError:
                                pass
                        break 
                        
                final_dataset.append({
                    'Name': search_name,
                    'Publishing Year': game_year,
                    'Description': desc,
                    'Average Score': avg_score,
                    'Number of Reviews': num_reviews
                })
                
                # Immediately update both tracking sets so Phase 1 knows about it next year
                existing_names.add(search_name.lower())
                normalized_existing_names.add(re.sub(r'[^a-z0-9]', '', search_name.lower()))
                
                print(f"| ✅ Score: {avg_score:.2f} | Total DB: {len(final_dataset)}")
                
                # 🌟 THE 10-SCRAPE AUTO-SAVE 🌟
                session_scrapes_count += 1
                if session_scrapes_count % 10 == 0:
                    pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
                    print(f"       💾 Auto-Save: 10 new games acquired. CSV safely overwritten.")
                
            elif res.status_code == 429:
                print("| 🚨 Rate Limited! Pausing for 30 seconds...")
                time.sleep(30)
            elif res.status_code == 403:
                print("| 🚨 403 Forbidden! Cloudflare blocked you.")
                time.sleep(30)
            else:
                print(f"| ⚠️ Failed Status: {res.status_code}")
                
        except Exception as e:
            print(f"| Error: {e}")

    # --- END OF YEAR CHECKPOINT ---
    pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
    
    cooldown = random.randint(15, 30)
    print(f"\n  💾 Checkpoint: {year} Complete & Saved. Total DB: {len(final_dataset)}")
    print(f"  ☕ Inter-year cooldown for {cooldown}s...")
    time.sleep(cooldown)

# ==========================================
# FINAL WRAP UP
# ==========================================
# One final save just in case the loop breaks before a multiple of 10 or end of year
pd.DataFrame(final_dataset).to_csv(OUTPUT_FILENAME, index=False)
print(f"\n🎉 SCRIPT FINISHED! Database now has {len(final_dataset)} total games in '{OUTPUT_FILENAME}'")

🚀 Starting the Year-by-Year Interleaved TTRPG Scraper (10-Scrape Auto-Save)...
📦 Loaded 1995 existing games from CSV.

📅 STARTING YEAR: 1985 | Target needed to hit 10k: 8005
  [PHASE 1] Harvesting Max 210 Links (Pre-Skipping Duplicates)...
    -> Scraping Page 1 ... | Added: 86  | Pre-Skipped: 14  | 🎯 Year Total: 86/210
    -> Scraping Page 2 ... | Added: 98  | Pre-Skipped: 2   | 🎯 Year Total: 184/210
    -> Scraping Page 3 ... | Added: 26  | Pre-Skipped: 0   | 🎯 Year Total: 210/210
    -> 🎯 Reached 210 limit for 1985.

  [PHASE 2] Scraping Details for 210 new games from 1985...
    -> 1/210: [1985] Gods Of Harn   ... | ✅ Score: 8.12 | Total DB: 1996
    -> 2/210: [1985] Stormbringer 2N... | ✅ Score: 7.00 | Total DB: 1997
    -> 3/210: [1985] Mystery Of The ... | ✅ Score: 5.27 | Total DB: 1998
    -> 4/210: [1985] Dragons Of War ... | ✅ Score: 6.47 | Total DB: 1999
    -> 5/210: [1985] Judge Dredd The... | ✅ Score: 6.89 | Total DB: 2000
    -> 6/210: [1985] The Kingdoms Of... | ✅ Score

KeyboardInterrupt: 